# 00. Dataset Metadata — GSE225845

This notebook extracts and standardizes the phenotypic metadata for **GSE225845**,
ensuring consistent sample labeling and reproducible downstream integration.
All relevant clinical and technical variables were parsed from the GEO records and stored in a compressed Parquet table, including age, race, sex, and tissue information.
Samples were categorized into three classes — **0 = normal**, **1 = normal-adjacent (adj_norm)**, **2 = tumor** — according to the *sample type* field.


**Source: GEO accession GSE225845, platform Illumina HumanMethylation450 BeadChip**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     00-dataset-metadata-GSE225845                      ║
# ║ Description: : Extracts and standardizes phenotypic metadata,    ║
# ║                ensuring consistent sample labeling and           ║
# ║                reproducible downstream integration.              ║
# ║ Dataset(s):   GSE225845                                          ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 10-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


## Libraries

In [ ]:
!pip install -q GEOparse polars pyarrow lz4


In [ ]:
import os
from pathlib import Path
from typing import Optional
import GEOparse
import polars as pl
import re


## 1. Builf pheno_GSE287331

In [ ]:
# BUILD PHENO.parquet FOR GSE225845 UNING GEOparse + Polars
# Columns:
#   - geo_accession
#   - sample_name
#   - source_name
#   - sample_type_raw
#   - tissue_type_raw
#   - age_at_surgery
#   - race
#   - sex
#   - idat_basename
#   - label_3class (0 = normal, 1 = benign/adjacent, 2 = tumor)

GSE_ID        = "GSE225845"  # GEO accession
DESTDIR       = "/kaggle/working/geoparse_cache"          # where to cache SOFT files
OUT_PHENO_PAR = "/kaggle/working/pheno_GSE225845.parquet" # output phenotype Parquet

Path(DESTDIR).mkdir(parents=True, exist_ok=True)
BETA_PARQUET_PATH = "/kaggle/input/gse225845-betas-parquet/GSE225845_betas.parquet"

def get_first(meta: dict, key: str) -> Optional[str]:
    """
    Safely extract the first element from a GEO metadata list.
    Returns None if key is missing or list is empty.
    """
    vals = meta.get(key, [])
    if not vals:
        return None
    return vals[0]


def extract_char(meta: dict, field_prefix: str) -> Optional[str]:
    """
    From metadata['characteristics_ch1'], extract the value of a field
    like 'sample type', 'age_at_surgery', 'race', 'Sex', etc.

    Looks for strings of the form 'field_prefix: value'.
    Matching is case-insensitive on the prefix.
    """
    for val in meta.get("characteristics_ch1", []):
        if val is None:
            continue
        lower = val.lower()
        if lower.startswith(field_prefix.lower() + ":"):
            # Split only on the first ':'
            return val.split(":", 1)[1].strip()
    return None


def map_sample_type_to_label(sample_type: Optional[str]) -> Optional[int]:
    """
    Map the raw 'sample type' string to a 3-class label:
      0 = normal
      1 = normal-adjacent (adj_norm)
      2 = breast cancer / tumor
    """
    if sample_type is None:
        return None

    s = sample_type.strip().lower()

    # --- explicit mapping for GSE225845 ---
    if s in {"normal"}:
        return 0
    if s in {"adj_norm"}:
        return 1
    if s in {"tumor"}:
        return 2
    return None

# 1) Download/load GSE with GEOparse
gse = GEOparse.get_GEO(
    geo=GSE_ID,
    destdir=DESTDIR,
    annotate_gpl=False,  # we don't need probe annotation here
    how="full",          # full SOFT
    silent=True,
)

# 2) Extract per-sample metadata into a list of dicts
rows: list[dict] = []

for gsm_name, gsm in gse.gsms.items():
    meta = gsm.metadata

    # geo_accession (GSM ID)
    geo_accession = gsm.get_accession()  # e.g. "GSM7057514"

    # sample_name from title
    sample_name = get_first(meta, "title") or geo_accession

    # source_name (usually 'frozen normal breast tissue', etc.)
    source_name = get_first(meta, "source_name_ch1")

    # Raw fields from 'characteristics_ch1'
    sample_type_raw = extract_char(meta, "sample type")
    tissue_type_raw = extract_char(meta, "tissue")  # if present as 'tissue: ...'

    age_at_surgery = extract_char(meta, "age_at_surgery")
    race           = extract_char(meta, "race")
    sex            = extract_char(meta, "sex")

    # IDAT basename from 'methylation id (basenames)'
    idat_basename = extract_char(meta, "methylation id (basenames)")

    # Map to 3-class label
    label_3class = map_sample_type_to_label(sample_type_raw)

    rows.append(
        {
            "geo_accession": geo_accession,
            "sample_name": sample_name,
            "source_name": source_name,
            "sample_type_raw": sample_type_raw,
            "tissue_type_raw": tissue_type_raw,
            "age_at_surgery": age_at_surgery,
            "race": race,
            "sex": sex,
            "idat_basename": idat_basename,
            "label_3class": label_3class,
        }
    )

# 3) Build Polars DataFrame and light typing
pheno = pl.DataFrame(rows)

# Basic cleaning / type casting
pheno = pheno.with_columns(
    # Cast label to small int
    pl.col("label_3class").cast(pl.Int8),
    # Age as numeric, if parseable
    pl.col("age_at_surgery")
      .str.replace_all(r"[^\d.]", "")  # remove any stray chars
      .cast(pl.Float32, strict=False)
      .alias("age_at_surgery"),
    # Normalize sex to upper-case (F/M/etc.)
    pl.col("sex").str.strip_chars().str.to_uppercase()
)

print(pheno.head())
print(pheno.select(
    "sample_type_raw",
    "tissue_type_raw",
    "age_at_surgery",
    "race",
    "sex",
    "label_3class"
).head(20))

print("\nValue counts for sample_type_raw:")
print(pheno["sample_type_raw"].value_counts())

print("\nValue counts for label_3class:")
print(pheno["label_3class"].value_counts())

# 4) Save to Parquet (LZ4)
pheno.write_parquet(
    OUT_PHENO_PAR,
    compression="lz4",
    statistics=True,
)

print(f"\n✅ Saved phenotype table to: {OUT_PHENO_PAR}")
print(f"Shape: {pheno.shape}")


## 2. Verification of sample count discrepancies and biological replicates

Since the number of samples retrieved from GEO did not match the total reported on the dataset’s web page, a series of duplicate and pairing checks was performed.
These analyses confirmed the presence of **biological replicates** (paired normal–tumor samples) rather than technical duplicates.
All samples were retained, but these pairings must be handled with caution when performing **training/validation/test splits** to avoid data leakage.


In [1]:
# FIND TECHNICAL DUPLICATES -> NONE 
pheno = pl.read_parquet("/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet")

# Exact duplicates of idat_basename
dupes_idat = (
    pheno.select("idat_basename")
         .filter(pl.col("idat_basename").is_not_null())
         .is_duplicated()
         .sum()
)
print(f"Duplicated idat_basename entries: {dupes_idat}")

# Duplicates by sample_name (different GSM but same name)
dupes_sample = (
    pheno.group_by("sample_name").len().filter(pl.col("len") > 1)
)
print(f"Duplicated sample_name groups:\n{dupes_sample}")

# Check for combinations (sample_type_raw + idat_basename)
dupes_combo = (
    pheno.group_by(["sample_type_raw", "idat_basename"]).len().filter(pl.col("len") > 1)
)
print(f"Duplicated combinations:\n{dupes_combo}")


Duplicated idat_basename entries: 0
Duplicated sample_name groups:
shape: (0, 2)
┌─────────────┬─────┐
│ sample_name ┆ len │
│ ---         ┆ --- │
│ str         ┆ u32 │
╞═════════════╪═════╡
└─────────────┴─────┘
Duplicated combinations:
shape: (0, 3)
┌─────────────────┬───────────────┬─────┐
│ sample_type_raw ┆ idat_basename ┆ len │
│ ---             ┆ ---           ┆ --- │
│ str             ┆ str           ┆ u32 │
╞═════════════════╪═══════════════╪═════╡
└─────────────────┴───────────────┴─────┘


In [3]:
# FIND BIOLOGICAL REPLICATES -> FIND!
# Extract any numerical codes (e.g., '92-6', '34', etc.)
pheno = pheno.with_columns(
    pl.col("sample_name")
      .str.extract(r"(\d+-\d+|\d+)")
      .alias("case_id_guess")
)

# Count how many times each code appears
counts = pheno.group_by("case_id_guess").len().filter(pl.col("len") > 1)
print(counts.sort("len", descending=True))


shape: (140, 2)
┌───────────────┬─────┐
│ case_id_guess ┆ len │
│ ---           ┆ --- │
│ str           ┆ u32 │
╞═══════════════╪═════╡
│ 18084         ┆ 2   │
│ 14064         ┆ 2   │
│ 12273         ┆ 2   │
│ 15050         ┆ 2   │
│ 27108         ┆ 2   │
│ …             ┆ …   │
│ 23891         ┆ 2   │
│ 12887         ┆ 2   │
│ 17952         ┆ 2   │
│ 25139         ┆ 2   │
│ 12788         ┆ 2   │
└───────────────┴─────┘
